# Experimental Framework: Per-Dataset Model Benchmarking
**Triage IQ — Predicting Hypertension from Emergency Department Triage Data**

This notebook implements the Phase 1 benchmarking pipeline described in the dissertation.  
Seven classifiers are evaluated on Dataset 1 (RIPAS ED triage data with an engineered hypertension label) using RandomizedSearchCV with 5-fold stratified cross-validation and SMOTE applied within each fold.

**Dataset used:** `kaggle_data/dataset1.csv`  
**Target variable:** Engineered from JNC-7 criteria — `(SBP ≥ 140) | (DBP ≥ 90)`


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os

from scipy.stats import randint, uniform, mannwhitneyu, chi2_contingency

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


## 2. Load and Prepare Dataset 1

Dataset 1 contains triage records with vital signs but no pre-labelled hypertension column.  
The target is engineered using JNC-7 blood pressure thresholds: **SBP ≥ 140 or DBP ≥ 90**.


In [ ]:
# Load dataset
df1 = pd.read_csv("kaggle_data/dataset1.csv", encoding='latin1', sep=None, engine='python')

# Cast relevant columns to numeric (coerce invalid entries to NaN)
for col in ["SBP", "DBP", "Age", "HR", "RR", "BT", "Saturation"]:
    df1[col] = pd.to_numeric(df1[col], errors="coerce")

# Engineer hypertension label (JNC-7 criteria)
df1["Hypertension"] = ((df1["SBP"] >= 140) | (df1["DBP"] >= 90)).astype(int)

# Selected features for modelling
selected_features = ["Age", "SBP", "DBP", "HR", "RR", "BT", "Saturation", "Pain", "NRS_pain"]

print("Dataset shape:", df1.shape)
print("\nClass distribution:")
print(df1["Hypertension"].value_counts())
print("\nClass proportions:")
print(df1["Hypertension"].value_counts(normalize=True).round(3))


## 3. Class Distribution

In [ ]:
df1["Hypertension"].value_counts().plot(kind="bar", color=["skyblue", "salmon"])
plt.title("Class Distribution — Dataset 1")
plt.xlabel("Hypertension (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Descriptive Statistics

In [ ]:
model_df = df1[selected_features + ["Hypertension"]].dropna().copy()
print("Modelling dataset shape:", model_df.shape)
display(model_df.describe().round(2))


## 5. Statistical Feature Analysis

Three tests are used to assess association between each feature and the hypertension label:
- **Spearman correlation** — monotonic association for numeric features  
- **Mann–Whitney U test** — non-parametric group comparison (hypertensive vs non-hypertensive)  
- **Chi-square test** — association for categorical features


In [ ]:
target = "Hypertension"
X = model_df[selected_features]
y = model_df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

# --- Spearman Correlation ---
spearman_corr = (
    model_df[numeric_features + [target]]
    .corr(method="spearman")[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
)

print("Spearman Correlation with Hypertension:")
print(spearman_corr.round(4))

plt.figure(figsize=(6, 5))
spearman_corr.plot(kind="barh")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Spearman Correlation — Dataset 1")
plt.xlabel("Spearman Correlation")
plt.tight_layout()
plt.show()


In [ ]:
# --- Mann–Whitney U Test ---
mann_results = []
for col in numeric_features:
    group0 = model_df[model_df[target] == 0][col].dropna()
    group1 = model_df[model_df[target] == 1][col].dropna()
    if len(group0) > 0 and len(group1) > 0:
        stat, p = mannwhitneyu(group0, group1, alternative="two-sided")
        mann_results.append({"Feature": col, "p_value": p})

mann_df = pd.DataFrame(mann_results).sort_values("p_value")
print("Mann–Whitney U Results — Dataset 1:")
print(mann_df.round(6))

plt.figure(figsize=(6, 5))
plt.barh(mann_df["Feature"], -np.log10(mann_df["p_value"]))
plt.xlabel("-log10(p-value)")
plt.title("Mann–Whitney U Test — Dataset 1")
plt.tight_layout()
plt.show()


In [ ]:
# --- Chi-Square Test (categorical features) ---
chi_results = []
for col in categorical_features:
    contingency = pd.crosstab(model_df[col], model_df[target])
    if contingency.shape[0] > 1:
        chi2, p, dof, _ = chi2_contingency(contingency)
        chi_results.append({"Feature": col, "Chi2": chi2, "p_value": p})

if chi_results:
    chi_df = pd.DataFrame(chi_results).sort_values("Chi2", ascending=False)
    print("Chi-Square Results — Dataset 1:")
    print(chi_df.round(4))

    plt.figure(figsize=(6, 4))
    plt.barh(chi_df["Feature"], chi_df["Chi2"])
    plt.xlabel("Chi-Square Statistic")
    plt.title("Chi-Square Association — Dataset 1")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No categorical features to test.")


## 6. Pre-Model Feature Importance (XGBoost)

A preliminary XGBoost model is fitted on all features to rank their importance before hyperparameter tuning.


In [ ]:
from xgboost import XGBClassifier

X_pm = model_df[selected_features]
y_pm = model_df["Hypertension"]

num_cols_pm = X_pm.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols_pm = X_pm.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor_pm = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols_pm),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols_pm)
])

xgb_pm_pipeline = Pipeline([
    ("preprocessor", preprocessor_pm),
    ("clf", XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                          random_state=42, eval_metric="logloss"))
])

xgb_pm_pipeline.fit(X_pm, y_pm)

feature_names_pm = xgb_pm_pipeline.named_steps["preprocessor"].get_feature_names_out()
importance_df = pd.DataFrame({
    "Feature": feature_names_pm,
    "Importance": xgb_pm_pipeline.named_steps["clf"].feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 15 Features — Pre-Model XGBoost (Dataset 1):")
display(importance_df.head(15))

plt.figure(figsize=(8, 6))
plt.barh(
    importance_df["Feature"][:15][::-1],
    importance_df["Importance"][:15][::-1]
)
plt.xlabel("Feature Importance Score")
plt.title("Top 15 Features — Pre-Model XGBoost (Dataset 1)")
plt.tight_layout()
plt.show()


## 7. Train/Test Split

A 70/30 stratified split is applied. SMOTE is applied **within** the training pipeline only, via `ImbPipeline`, to prevent data leakage into the test set.


In [ ]:
X = model_df[selected_features]
y = model_df["Hypertension"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train.value_counts())


## 8. Preprocessing Pipeline

Numerical features are median-imputed and scaled; categorical features are mode-imputed and one-hot encoded.


In [ ]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

print("Numeric features:", num_cols)
print("Categorical features:", cat_cols)


## 9. Model and Hyperparameter Grid Definitions

In [ ]:
models = {
    "LogisticRegression": (
        LogisticRegression(max_iter=2000),
        {"clf__C": uniform(0.01, 10), "clf__solver": ["liblinear", "saga"]}
    ),

    "DecisionTree": (
        DecisionTreeClassifier(),
        {
            "clf__max_depth": randint(3, 6),
            "clf__min_samples_split": randint(5, 10),
            "clf__min_samples_leaf": randint(2, 5)
        }
    ),

    "RandomForest": (
        RandomForestClassifier(),
        {
            "clf__n_estimators": randint(100, 500),
            "clf__max_depth": [None, 5, 10, 20],
            "clf__min_samples_split": randint(2, 20),
            "clf__min_samples_leaf": randint(1, 10)
        }
    ),

    "SVM": (
        SVC(probability=True),
        {
            "clf__C": uniform(0.1, 20),
            "clf__kernel": ["rbf"],
            "clf__gamma": ["scale", "auto"]
        }
    ),

    "GradientBoosting": (
        GradientBoostingClassifier(),
        {
            "clf__n_estimators": randint(50, 200),
            "clf__learning_rate": uniform(0.01, 0.2)
        }
    ),

    "MLP": (
        MLPClassifier(max_iter=2000),
        {
            "clf__hidden_layer_sizes": [(50,), (100,), (100, 50)],
            "clf__alpha": uniform(0.0001, 0.01)
        }
    ),

    "XGBoost": (
        XGBClassifier(eval_metric="logloss", use_label_encoder=False),
        {
            "clf__n_estimators": randint(100, 400),
            "clf__max_depth": randint(3, 10),
            "clf__learning_rate": uniform(0.01, 0.2),
            "clf__subsample": uniform(0.6, 0.4),
            "clf__colsample_bytree": uniform(0.6, 0.4),
            "clf__gamma": uniform(0, 5)
        }
    ),
}


## 10. Baseline Model Evaluation (No Hyperparameter Tuning)

Each model is fitted once with default parameters and SMOTE applied within the pipeline.  
This provides a performance floor before tuning.


In [ ]:
baseline_results = []

for name, (model, _) in models.items():
    print(f"Training baseline {name}...")

    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("clf", model)
    ])
    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe["clf"], "predict_proba") else None

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    baseline_results.append({
        "Model": name,
        "Baseline Accuracy": round(acc, 3),
        "Baseline ROC-AUC": round(auc, 3) if auc else None
    })
    print(f"  Accuracy: {acc:.3f}  |  ROC-AUC: {auc:.3f}\n")

baseline_df = pd.DataFrame(baseline_results).sort_values("Baseline ROC-AUC", ascending=False)
print("\nBaseline Results Summary:")
display(baseline_df)


## 11. Hyperparameter Tuning — RandomizedSearchCV

`RandomizedSearchCV` with 5-fold stratified cross-validation is run for each model.  
SMOTE is applied inside each fold via `ImbPipeline` to prevent leakage.  
Optimisation metric: **ROC-AUC** (30 iterations per model).


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
best_models = {}

for name, (model, param_grid) in models.items():
    print(f"\nRunning RandomizedSearchCV for {name}...")

    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("clf", model)
    ])

    search = RandomizedSearchCV(
        pipe,
        param_distributions=param_grid,
        n_iter=30,
        scoring="roc_auc",
        cv=cv,
        random_state=99,
        verbose=0,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_models[name] = search

    best_pipe = search.best_estimator_
    y_pred    = best_pipe.predict(X_test)
    y_proba   = (
        best_pipe.predict_proba(X_test)[:, 1]
        if hasattr(best_pipe["clf"], "predict_proba") else None
    )

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    results.append({
        "Model": name,
        "Best CV Score (RandomizedSearchCV)": round(search.best_score_, 3),
        "CV Accuracy": round(acc, 3),
        "CV ROC-AUC": round(auc, 3) if auc else None
    })

    print(f"  Best CV ROC-AUC: {search.best_score_:.3f}  |  Test Accuracy: {acc:.3f}  |  Test ROC-AUC: {auc:.3f}")
    print(classification_report(y_test, y_pred))
    print("-" * 70)

results_df = pd.DataFrame(results).sort_values("CV ROC-AUC", ascending=False)
print("\nModel Performance Summary — Dataset 1:")
display(results_df)


## 12. Model Performance Comparison

In [ ]:
# Table plot
fig, ax = plt.subplots(figsize=(10, len(results_df) * 0.5 + 1))
ax.axis("off")

tbl = ax.table(
    cellText=results_df.values,
    colLabels=results_df.columns,
    cellLoc="center",
    loc="center"
)
for j in range(len(results_df.columns)):
    tbl[0, j].set_facecolor("#40466e")
    tbl[0, j].set_text_props(color="w", weight="bold")
for i in range(1, len(results_df) + 1):
    color = "#f9f9f9" if i % 2 == 1 else "#ffffff"
    for j in range(len(results_df.columns)):
        tbl[i, j].set_facecolor(color)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.5)
plt.title("Model Performance — Dataset 1 (RandomizedSearchCV)", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

# Bar chart
plot_df = results_df.melt(id_vars="Model", value_vars=["CV Accuracy", "CV ROC-AUC"],
                           var_name="Metric", value_name="Score")
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric", palette="Blues_d")
plt.title("Model Performance Comparison — Dataset 1")
plt.ylim(0.85, 1.05)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 13. Best Hyperparameters (RandomizedSearchCV)

In [ ]:
rows = []
for model_name, search in best_models.items():
    for param, value in search.best_params_.items():
        rows.append({
            "Model": model_name,
            "Parameter": param.replace("clf__", ""),
            "Best Value": value
        })

best_params_df = pd.DataFrame(rows)

# Suppress repeated model names for readability
best_params_df["Model_display"] = best_params_df["Model"]
best_params_df.loc[
    best_params_df["Model_display"] == best_params_df["Model_display"].shift(1),
    "Model_display"
] = ""

fig, ax = plt.subplots(figsize=(10, len(best_params_df) * 0.4 + 1))
ax.axis("off")

tbl = ax.table(
    cellText=best_params_df[["Model_display", "Parameter", "Best Value"]].values,
    colLabels=["Model", "Parameter", "Best Value"],
    cellLoc="center",
    loc="center"
)
for j in range(3):
    tbl[0, j].set_facecolor("#40466e")
    tbl[0, j].set_text_props(color="w", weight="bold")
for i in range(1, len(best_params_df) + 1):
    color = "#f9f9f9" if i % 2 == 1 else "#ffffff"
    for j in range(3):
        tbl[i, j].set_facecolor(color)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.5)
plt.title("RandomizedSearchCV Best Parameters — Dataset 1", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

display(best_params_df[["Model", "Parameter", "Best Value"]])


## 14. Save Results to Excel

In [ ]:
os.makedirs("results", exist_ok=True)

# Performance table
performance_file = "results/model_performance_dataset1.xlsx"
results_df.to_excel(performance_file, index=False)
print(f"Model performance saved: {performance_file}")

# Best parameters table
params_file = "results/best_params_dataset1.xlsx"
best_params_df[["Model", "Parameter", "Best Value"]].to_excel(params_file, index=False)
print(f"Best parameters saved:   {params_file}")


## 15. SHAP Explainability — XGBoost (Dataset 1)

SHAP (SHapley Additive Explanations) is used to identify which features drive the XGBoost predictions.  
Both global (summary) and local (waterfall) explanations are shown.


In [ ]:
import xgboost as xgb

# Retrieve the best XGBoost pipeline
xgb_search  = best_models["XGBoost"]
final_pipeline = xgb_search.best_estimator_

# Transform the full feature matrix
X_transformed = final_pipeline.named_steps["preprocessor"].transform(X)
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

# Recover feature names after preprocessing
numeric_features_out = num_cols
cat_features_out = (
    final_pipeline.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(cat_cols)
) if cat_cols else []

feature_names = list(numeric_features_out) + list(cat_features_out)
print(f"Total features after encoding: {len(feature_names)}")

# Plot XGBoost built-in feature importance
xgb.plot_importance(
    final_pipeline.named_steps["clf"],
    max_num_features=10,
    importance_type="gain"
)
plt.title("Top 10 Most Important Features — XGBoost (Dataset 1)")
plt.tight_layout()
plt.show()


In [ ]:
# SHAP global summary
explainer   = shap.Explainer(final_pipeline.named_steps["clf"], X_transformed)
shap_values = explainer(X_transformed)

# Determine correct SHAP values (binary vs multiclass)
shap_array = shap_values.values
if shap_array.ndim == 3:
    shap_class1 = shap_array[:, :, 1]
else:
    shap_class1 = shap_array

print("Global SHAP Summary Plot (positive class — Hypertension = 1):")
shap.summary_plot(shap_class1, X_transformed, feature_names=feature_names)


In [ ]:
# Top 10 SHAP features by mean absolute value
mean_abs_shap = np.abs(shap_class1).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP|": mean_abs_shap
}).sort_values("Mean |SHAP|", ascending=False)

print("Top 10 Most Important Features (SHAP):")
display(shap_importance.head(10))

# Bar chart
top10 = shap_importance.head(10)
plt.figure(figsize=(7, 5))
plt.barh(top10["Feature"][::-1], top10["Mean |SHAP|"][::-1], color="steelblue")
plt.xlabel("Mean |SHAP Value|")
plt.title("Top 10 SHAP Feature Importance — Dataset 1")
plt.tight_layout()
plt.show()


In [ ]:
# Local explanation — first sample
print("Local SHAP Explanation (first test sample):")
shap.plots.waterfall(shap_values[0, :, 1] if shap_values.values.ndim == 3 else shap_values[0], max_display=10)
